In [ ]:
import logging
from typing import List, Dict
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import google.generativeai as genai
import nest_asyncio
import uvicorn
from getpass import getpass  # do bezpiecznego wpisywania klucza

# Klucz API, zaimplementowałem to w ten sposób, żeby nie było widać klucza w kodzie
GEMINI_API_KEY = getpass("Wprowadź swój klucz API Gemini: ")

# Konfiguracja Gemini
genai.configure(api_key=GEMINI_API_KEY)

# Wykorzystałem proponowany gemini 2.5 flash
MODEL_NAME = "gemini-2.5-flash"

# Konfiguracja parametrów (trochę randomowo)
MAX_CONTEXT_MESSAGES = 20
TEMPERATURE = 0.7
TOP_P = 0.9
TOP_K = 40
MAX_TOKENS = 512

# Plik z logami
logging.basicConfig(
    filename="chatbot.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Klasy z modelami danych Request i Response
class ChatRequest(BaseModel):
    session_id: str
    message: str

class ChatResponse(BaseModel):
    response: str

# Główna klasa Chatbota
class Chatbot:
    def __init__(self):
        self.sessions: Dict[str, List[Dict[str, str]]] = {}
        self.model = genai.GenerativeModel(MODEL_NAME)
        # prosty prompt systemowy (bez niego dawał mi bardzo długie odpowiedzi)
        self.system_prompt = "Jesteś pomocnym chatbotem. Odpowiadaj jasno i zwięźle." 

    def _init_session(self, session_id: str):
        if session_id not in self.sessions:
            self.sessions[session_id] = [{"role": "system", "content": self.system_prompt}]

    def _trim_context(self, session_id: str):
        history = self.sessions[session_id]
        if len(history) > MAX_CONTEXT_MESSAGES:
            self.sessions[session_id] = [history[0]] + history[-(MAX_CONTEXT_MESSAGES - 1):]

    def _format_history(self, history: List[Dict[str, str]]) -> str:
        formatted = ""
        for msg in history:
            formatted += f"{msg['role'].upper()}: {msg['content']}\n"
        return formatted

    def chat(self, session_id: str, user_message: str) -> str:
        try:
            self._init_session(session_id)
            self.sessions[session_id].append({"role": "user", "content": user_message})
            self._trim_context(session_id)
            history_text = self._format_history(self.sessions[session_id])

            response = self.model.generate_content(
                history_text,
                generation_config={
                    "temperature": TEMPERATURE,
                    "top_p": TOP_P,
                    "top_k": TOP_K,
                    "max_output_tokens": MAX_TOKENS
                }
            )

            bot_reply = response.text
            self.sessions[session_id].append({"role": "assistant", "content": bot_reply})

            logging.info(f"[{session_id}] USER: {user_message}")
            logging.info(f"[{session_id}] BOT: {bot_reply}")

            return bot_reply

        except Exception as e:
            logging.error(f"Error in session {session_id}: {str(e)}")
            raise HTTPException(status_code=500, detail="Internal error")

In [ ]:
# FastAPI aplikacja 
app = FastAPI()
chatbot = Chatbot()

@app.post("/chat", response_model=ChatResponse)
def chat_endpoint(req: ChatRequest):
    if not req.message:
        raise HTTPException(status_code=400, detail="Empty message")
    response = chatbot.chat(req.session_id, req.message)
    return ChatResponse(response=response)

# uruchomienie serwera
nest_asyncio.apply()

# test czy zapmiętuje cytat w ramach danej sesji - test123
response = chatbot.chat("test123", "Cześć, zapamiętaj cytat: To be or not to be, that is the question")
print(response)

Zapamiętałem: "To be or not to be, that is the question".


In [ ]:
# sprawdzam co zapamiętał
response = chatbot.chat("test123", "Co zapamiętałeś o cytacie?")
print(response)

Zapamiętałem cytat: "To be or not to be, that is the question".


In [ ]:
# Wyświetlenie całej historii sesji test123
for msg in chatbot.sessions["test123"]:
    print(f"{msg['role']}: {msg['content']}")

system: Jesteś pomocnym chatbotem. Odpowiadaj jasno i zwięźle.
user: Cześć, zapamiętaj cytat: To be or not to be, that is the question
assistant: Zapamiętałem: "To be or not to be, that is the question".
user: Co zapamiętałeś o cytacie?
assistant: Zapamiętałem cytat: "To be or not to be, that is the question".
